> Legacy/optional workflow. For author SLM-SQL + CSC baseline and CSC LoRA research, use **09 → 10 → 11** and `docs/CSC_LORA.md`. This notebook is not the canonical comparison against the author system.

# Phase 8: reproducible report

This notebook reads only `results/final/*.json`. Every table row keeps its source filename and SHA-256 hash; no score is entered manually.

In [ ]:
from pathlib import Path
import csv
import hashlib
import json
import sys

_candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents, Path('/kaggle/working/SLM-PRJ')]
ROOT = next((p for p in _candidates if (p / 'pyproject.toml').is_file() and (p / 'benchmarks').is_dir()), Path('/kaggle/working/SLM-PRJ'))
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
FINAL_DIR = ROOT / 'results' / 'final'
REPORT_DIR = FINAL_DIR / 'report'
REPORT_DIR.mkdir(parents=True, exist_ok=True)
from evaluation.final_eval import load_json, sha256_path, write_json
from research.pareto import plot_pareto, save_pareto_report

all_source_files = [{'path': str(path), 'sha256': sha256_path(path)} for path in sorted(FINAL_DIR.glob('*.json'))]
report_artifacts = {'summary_table.json', 'seed_summary.json', 'mcnemar_table.json', 'pareto.json', 'provenance.json'}
source_files = [item for item in all_source_files if Path(item['path']).name not in report_artifacts and Path(item['path']).name != 'error_sample_100.json' and 'mcnemar' not in Path(item['path']).name]
mcnemar_files = [item for item in all_source_files if Path(item['path']).name not in report_artifacts and 'mcnemar' in Path(item['path']).name]
print({'source_json_files': len(all_source_files), 'score_files': len(source_files), 'directory': str(FINAL_DIR)})

In [ ]:
rows = []
for source in source_files:
    path = Path(source['path'])
    report = load_json(path)
    if 'execution_accuracy' not in report or 'questions' not in report:
        continue
    rows.append({
        'name': report.get('variant', path.stem),
        'source_file': path.name,
        'source_sha256': source['sha256'],
        'questions': report['questions'],
        'ex': report['execution_accuracy'],
        'ex_ci95_lower': report.get('accuracy_bootstrap_95', {}).get('lower'),
        'ex_ci95_upper': report.get('accuracy_bootstrap_95', {}).get('upper'),
        'avg_samples': report.get('average_candidates', 0.0),
        'avg_seconds': report.get('average_latency_ms', 0.0) / 1000.0,
        'seed': report.get('seed', report.get('config', {}).get('seed')),
        'config_sha256': hashlib.sha256(json.dumps(report.get('config', {}), sort_keys=True, ensure_ascii=False).encode()).hexdigest(),
        'protocol_signature': report.get('provenance', {}).get('protocol_signature'),
    })

identities = {}
for row in rows:
    key = (row['name'], row['seed'])
    identity = (row['config_sha256'], row['protocol_signature'])
    if key in identities:
        previous = identities[key]
        qualifier = 'mixed-config' if previous != identity else 'duplicate'
        raise ValueError(f'Duplicate or {qualifier} seed reports detected for {key}; clean results/final before aggregation.')
    identities[key] = identity

table_payload = {'rows': rows, 'provenance': source_files, 'note': 'Rows are derived only from final JSON artifacts.'}
write_json(REPORT_DIR / 'summary_table.json', table_payload)
if rows:
    with (REPORT_DIR / 'summary_table.csv').open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
print({'rows': len(rows), 'output': str(REPORT_DIR / 'summary_table.json')})
import statistics
from collections import defaultdict
seed_groups = defaultdict(list)
for row in rows:
    seed_groups[row['name']].append(row)
seed_summary = []
for name, group in seed_groups.items():
    values = [r['ex'] for r in group]
    seed_summary.append({'name': name, 'n_seeds': len(group),
                         'seeds': [r['seed'] for r in group],
                         'ex_mean': statistics.mean(values),
                         'ex_std': statistics.stdev(values) if len(values) > 1 else None})
write_json(REPORT_DIR / 'seed_summary.json', {'rows': seed_summary})


In [ ]:
pairwise_rows = []
for source in mcnemar_files:
    path = Path(source['path'])
    payload = load_json(path)
    pairwise_rows.append({**payload.get('mcnemar', {}), 'source_file': path.name, 'source_sha256': source['sha256']})
write_json(REPORT_DIR / 'mcnemar_table.json', {'rows': pairwise_rows, 'provenance': mcnemar_files})
print({'mcnemar_rows': len(pairwise_rows), 'output': str(REPORT_DIR / 'mcnemar_table.json')})

In [ ]:
if rows:
    save_pareto_report(rows, REPORT_DIR / 'pareto.json', ['avg_samples', 'avg_seconds'])
    plot_pareto(rows, 'avg_samples', 'ex', REPORT_DIR / 'pareto_samples.png')
    plot_pareto(rows, 'avg_seconds', 'ex', REPORT_DIR / 'pareto_seconds.png')
    print('Wrote Pareto tables and figures from the same source rows.')
else:
    print('No final evaluation reports found; no figure or score was generated.')

In [ ]:
# Human-readable provenance index for the report directory.
index = {
    'generated_from': all_source_files,
    'artifacts': [str(path.relative_to(ROOT)) for path in sorted(REPORT_DIR.iterdir())],
    'warning': 'Metrics are absent when their source JSON artifact is absent; no placeholder zero is used.',
}
write_json(REPORT_DIR / 'provenance.json', index)
print(json.dumps(index, ensure_ascii=False, indent=2))